# W33 · Isaac Lab 自定义 RL 任务

> 阶段四第 3 周。本周目标：把你在阶段一写过的 `DroneHoverEnv`（一维定高）的设计经验，
> 迁移成 Isaac Lab 里的 3D 四旋翼悬停任务。两种 workflow 各走一遍。

## 学习目标

1. 把 Gymnasium 自定义环境的要素（obs/action/reward/termination/reset）映射到 Isaac Lab 的 Manager 术语；
2. 用 **Manager-based** workflow 拼装一个自定义任务（以 CartPole 为模板，改奖励与终止）；
3. 用 **Direct** workflow 写四旋翼悬停任务骨架（对标官方 `Isaac-Quadcopter-Direct-v0`）；
4. 注册自定义任务并用 rsl_rl 启动训练；
5. 掌握自定义任务的调试清单。

## ⚠️ 运行前提

所有代码 cell 均需 **NVIDIA GPU + 按 `docs/phase4_isaac.md` 装好的 Isaac Lab**，
本机一律**保持未执行**。它们是可直接落地的模板：在 Isaac Lab 环境中按 cell 顺序
存成 `.py` 文件即可运行（落地路径见各 cell 注释）。

## 1. 设计迁移：DroneHoverEnv → Isaac Lab

阶段一的一维悬停环境（`src/robot_rl_learn/phase1_sb3/envs.py`）要素回顾：
观测 `[高度误差, 垂直速度]`，动作 `推力∈[-1,1]`，奖励 `-err² - 0.1·v² - 0.01·u²`，
终止 `|err|>3`，reset 随机初始高度。

3D 版映射表：

| DroneHoverEnv（Gym） | Isaac Lab（Manager / Direct） | 变化 |
|---|---|---|
| obs `[err_z, vz]`（numpy） | 观测 term：本体位置/姿态/线速度/角速度（GPU tensor，shape `(num_envs, 12)`） | 标量→批量张量 |
| action 标量推力 | 4 旋翼推力（Direct：`_apply_action` 里施加外力/力矩） | 1 维→4 维 |
| reward 三个平方项 | RewardsCfg：hover 误差/姿态/能耗各一个 `RewTerm` | 加权求和，权重即配置 |
| 终止 `\|err\|>3` | TerminationsCfg：`base_contact`/高度出界 + `time_out` | done 分 terminated/truncated |
| reset 全量归零 | EventCfg：`mode="reset"` 的随机初始状态 | 部分 reset（按 env_id） |
| 手写欧拉积分 | PhysX GPU 刚体动力学 + USD 资产（Crazyflie） | 不用写动力学 |

> 💡 核心观念转变：你不再「实现物理」，而是「**描述 MDP**」——物理交给 PhysX，
> 你负责观测、奖励、终止、随机化这四件 RL 工程师真正该操心的事。

## 2. 工程结构：任务放在哪

Isaac Lab 的任务是 Python 包（extension）。最省事的起点是官方模板
[IsaacLabExtensionTemplate](https://github.com/isaac-sim/IsaacLabExtensionTemplate)，
或在 `IsaacLab/source/isaaclab_tasks/` 旁边建自己的包。典型结构：

```
my_tasks/
├── my_tasks/
│   ├── tasks/
│   │   ├── cartpole_custom/          # Manager-based 示例
│   │   │   ├── __init__.py           # gym.register 在这里
│   │   │   ├── cartpole_env_cfg.py   # 环境配置（本 notebook Part A）
│   │   │   ├── mdp/
│   │   │   │   └── rewards.py        # 自定义奖励函数
│   │   │   └── agents/
│   │   │       └── rsl_rl_ppo_cfg.py # PPO 超参（下周 W34 细讲）
│   │   └── quadcopter_hover/         # Direct 示例
│   │       ├── __init__.py
│   │       ├── quadcopter_hover_env.py
│   │       └── agents/rsl_rl_ppo_cfg.py
└── setup.py / pyproject.toml
```

## 3. Part A：Manager-based —— 改造 CartPole

下面 4 个 cell 是一个完整可落地的 Manager-based 任务。
与官方 `Isaac-Cartpole-v0` 的差别在于：**奖励换成自定义函数、终止阈值收紧**——
这正是「改配置 + 写少量函数」的典型工作量。

In [ ]:
# 文件：tasks/cartpole_custom/cartpole_env_cfg.py  (第 1/4 部分：Scene + Actions)
# 前提：Isaac Lab 环境；未执行。
from isaaclab.assets import ArticulationCfg, AssetBaseCfg
from isaaclab.scene import InteractiveSceneCfg
from isaaclab.utils import configclass
import isaaclab.sim as sim_utils
from isaaclab.utils.assets import ISAAC_NUCLEUS_DIR

import isaaclab.envs.mdp as mdp
from isaaclab.managers import SceneEntityCfg

from isaaclab_assets import CARTPOLE_CFG  # 官方资产：USD 模型 + 关节驱动配置


@configclass
class MyCartpoleSceneCfg(InteractiveSceneCfg):
    """场景：地面 + 顶灯 + 一辆 CartPole。num_envs 决定并行份数。"""

    ground = AssetBaseCfg(
        prim_path="/World/defaultGroundPlane",
        spawn=sim_utils.GroundPlaneCfg(),
    )
    dome_light = AssetBaseCfg(
        prim_path="/World/Light",
        spawn=sim_utils.DomeLightCfg(intensity=3000.0, color=(0.75, 0.75, 0.75)),
    )
    # {ENV_REGEX_NS} 会展开成 /World/envs/env_0, env_1, ... —— 一份配置，N 份实例
    robot: ArticulationCfg = CARTPOLE_CFG.replace(prim_path="{ENV_REGEX_NS}/Robot")


@configclass
class ActionsCfg:
    """动作：对滑轨关节施加力（effort）。scale 把网络输出 [-1,1] 放大到牛顿量级。"""

    joint_effort = mdp.JointEffortActionCfg(
        entity_name="robot", joint_names=["slider_to_cart"], scale=100.0
    )

In [ ]:
# 文件：tasks/cartpole_custom/cartpole_env_cfg.py  (第 2/4 部分：Observations + 自定义奖励函数)
# 前提：Isaac Lab 环境；未执行。
import torch

from isaaclab.envs import ManagerBasedRLEnv
from isaaclab.managers import ObservationGroupCfg as ObsGroup
from isaaclab.managers import ObservationTermCfg as ObsTerm
from isaaclab.managers import SceneEntityCfg
from isaaclab.utils import configclass

import isaaclab.envs.mdp as mdp


@configclass
class ObservationsCfg:
    """策略观测：两个关节（滑轨、摆杆）的位置与速度，拼成一个 4 维向量。"""

    @configclass
    class PolicyCfg(ObsGroup):
        joint_pos_rel = ObsTerm(func=mdp.joint_pos_rel)
        joint_vel_rel = ObsTerm(func=mdp.joint_vel_rel)

        def __post_init__(self):
            self.enable_corruption = False  # True 可加观测噪声（Sim2Real 用）
            self.concatenate_terms = True

    policy: PolicyCfg = PolicyCfg()


# ---- 自定义奖励函数（写在 mdp/rewards.py 里更规范，这里为教学内联展示）----
# 约定：输入 env + 可选 asset_cfg，输出 shape (num_envs,) 的 tensor
def alive(env: ManagerBasedRLEnv) -> torch.Tensor:
    """每活一步给固定奖励，鼓励不倒。"""
    return torch.ones(env.num_envs, device=env.device)


def terminating(env: ManagerBasedRLEnv) -> torch.Tensor:
    """终止时给惩罚（本步 reset 因 terminated 触发）。"""
    return env.reset_terminated.float()


def pole_angle(env: ManagerBasedRLEnv, asset_cfg: SceneEntityCfg = SceneEntityCfg("robot")) -> torch.Tensor:
    """摆杆偏离竖直的角度（joint 1），越小越好。"""
    asset = env.scene[asset_cfg.name]
    return torch.abs(asset.data.joint_pos[:, 1])


def cart_pos(env: ManagerBasedRLEnv, asset_cfg: SceneEntityCfg = SceneEntityCfg("robot")) -> torch.Tensor:
    """小车偏离原点的距离（joint 0），越小越好。"""
    asset = env.scene[asset_cfg.name]
    return torch.abs(asset.data.joint_pos[:, 0])

In [ ]:
# 文件：tasks/cartpole_custom/cartpole_env_cfg.py  (第 3/4 部分：Rewards + Terminations + Events)
# 前提：Isaac Lab 环境；未执行。
from isaaclab.managers import RewardTermCfg as RewTerm
from isaaclab.managers import TerminationTermCfg as DoneTerm
from isaaclab.managers import EventTermCfg as EventTerm
from isaaclab.managers import SceneEntityCfg
from isaaclab.utils import configclass

import isaaclab.envs.mdp as mdp


@configclass
class RewardsCfg:
    """总奖励 = Σ(weight × func)，weight 就是你调参的主战场。"""

    alive = RewTerm(func=alive, weight=1.0)
    terminating = RewTerm(func=terminating, weight=-2.0)
    pole_angle = RewTerm(func=pole_angle, weight=-1.0)   # 摆角惩罚
    cart_pos = RewTerm(func=cart_pos, weight=-0.5)       # 偏离中心惩罚
    joint_vel = RewTerm(func=mdp.joint_vel_l2, weight=-1e-3, params={"asset_cfg": SceneEntityCfg("robot")})


@configclass
class TerminationsCfg:
    """done 判定：超时（truncated）与小车出界（terminated）。"""

    time_out = DoneTerm(func=mdp.time_out, time_out=True)
    cart_out_of_bounds = DoneTerm(
        func=mdp.joint_pos_out_of_manual_limit,
        params={"asset_cfg": SceneEntityCfg("robot", joint_names=["slider_to_cart"]),
                "bounds": (-3.0, 3.0)},
    )


@configclass
class EventCfg:
    """reset 时的随机化：初始状态扰动（域随机化的最小形态）。"""

    reset_cart_position = EventTerm(
        func=mdp.reset_joints_by_offset,
        mode="reset",
        params={"asset_cfg": SceneEntityCfg("robot", joint_names=["slider_to_cart"]),
                "position_range": (-1.0, 1.0),
                "velocity_range": (-0.1, 0.1)},
    )
    reset_pole_position = EventTerm(
        func=mdp.reset_joints_by_offset,
        mode="reset",
        params={"asset_cfg": SceneEntityCfg("robot", joint_names=["pole_to_cart"]),
                "position_range": (-0.25, 0.25),
                "velocity_range": (-0.1, 0.1)},
    )

In [ ]:
# 文件：tasks/cartpole_custom/cartpole_env_cfg.py  (第 4/4 部分：组装 + 注册)
# 前提：Isaac Lab 环境；未执行。
import gymnasium as gym

from isaaclab.envs import ManagerBasedRLEnvCfg
from isaaclab.utils import configclass


@configclass
class MyCartpoleEnvCfg(ManagerBasedRLEnvCfg):
    """总装：把上面各 Manager 配置挂到环境配置上。"""

    scene: MyCartpoleSceneCfg = MyCartpoleSceneCfg(num_envs=4096, env_spacing=4.0)
    observations: ObservationsCfg = ObservationsCfg()
    actions: ActionsCfg = ActionsCfg()
    rewards: RewardsCfg = RewardsCfg()
    terminations: TerminationsCfg = TerminationsCfg()
    events: EventCfg = EventCfg()

    def __post_init__(self):
        self.decimation = 2               # 物理 2 步 = 控制 1 步（120 Hz 物理 → 60 Hz 控制）
        self.episode_length_s = 5.0
        self.sim.dt = 1.0 / 120.0
        self.sim.render_interval = self.decimation


# 注册进 Gym（通常放在 tasks/cartpole_custom/__init__.py）
gym.register(
    id="My-Cartpole-v0",
    entry_point="isaaclab.envs:ManagerBasedRLEnv",
    disable_env_checker=True,
    kwargs={
        "env_cfg_entry_point": f"{__name__}:MyCartpoleEnvCfg",
        # PPO 超参配置见 W34；rsl_rl 训练时通过此 entry point 读取
        # "rsl_rl_cfg_entry_point": "<agents 模块路径>:MyCartpolePPORunnerCfg",
    },
)

# 训练命令（在 IsaacLab/ 目录下执行）：
# ./isaaclab.sh -p scripts/reinforcement_learning/rsl_rl/train.py \
#     --task=My-Cartpole-v0 --num_envs=4096 --headless
# 预期输出：logs/rsl_rl/ 下生成实验目录，mean_reward 几十轮内爬升并收敛
# （CartPole 是简单任务，GPU 上通常 1-2 分钟收敛）。

## 4. Part B：Direct —— 四旋翼悬停（DroneHover 的 3D 版）

Direct workflow 把 MDP 全部写进一个 `DirectRLEnv` 子类。官方自带
`Isaac-Quadcopter-Direct-v0`（Crazyflie 定点飞行），下面骨架与之同构，
把阶段一的一维悬停升级为 **3D 位置 + 姿态** 的悬停：

- **观测（12 维）**：目标相对位置（3）+ 线速度（3）+ 姿态四元数/投影重力（3~4）+ 角速度（3）
- **动作（4 维）**：4 个旋翼归一化推力 → 总推力 + 三轴力矩
- **奖励**：位置误差、姿态误差、动作平滑、能耗
- **终止**：坠毁（触地）/ 飞太远；超时截断

> 骨架中 `...` 处请对照官方 `source/isaaclab_tasks/isaaclab_tasks/direct/quadcopter/quadcopter_env.py`
> 补全——读官方实现本身就是本周最重要的作业（练习 2）。

In [ ]:
# 文件：tasks/quadcopter_hover/quadcopter_hover_env.py（Direct 骨架；未执行）
import torch
from isaaclab.envs import DirectRLEnv, DirectRLEnvCfg
from isaaclab.scene import InteractiveSceneCfg
from isaaclab.sim import SimulationCfg
from isaaclab.utils import configclass

from isaaclab_assets import CRAZYFLIE_CFG  # Crazyflie 2.x 四旋翼（27 g）


@configclass
class QuadcopterHoverEnvCfg(DirectRLEnvCfg):
    episode_length_s = 10.0
    decimation = 2                        # 100 Hz 物理 → 50 Hz 控制
    action_space = 4                      # 4 旋翼
    observation_space = 12
    state_space = 0                       # 非对称 actor-critic 才用
    sim: SimulationCfg = SimulationCfg(dt=1 / 100, render_interval=decimation)
    scene: InteractiveSceneCfg = InteractiveSceneCfg(
        num_envs=4096, env_spacing=2.5, replicate_physics=True
    )
    robot = CRAZYFLIE_CFG.replace(prim_path="/World/envs/env_.*/Robot")
    thrust_to_weight = 1.9                # 推重比
    moment_scale = 0.01                   # 力矩缩放


class QuadcopterHoverEnv(DirectRLEnv):
    cfg: QuadcopterHoverEnvCfg

    def __init__(self, cfg, render_mode=None, **kwargs):
        super().__init__(cfg, render_mode, **kwargs)
        # 动作缓冲与外力/力矩缓冲（物理 step 期间反复施加，每步只写一次即可）
        self._actions = torch.zeros(self.num_envs, 4, device=self.device)
        self._thrust = torch.zeros(self.num_envs, 1, 3, device=self.device)
        self._moment = torch.zeros(self.num_envs, 1, 3, device=self.device)
        # 目标点：z=1m 悬停
        self._desired_pos = torch.tensor([0.0, 0.0, 1.0], device=self.device)

    def _setup_scene(self):
        self._robot = self.scene["robot"]  # 由 InteractiveScene 按 cfg 生成

    def _pre_physics_step(self, actions: torch.Tensor):
        # 网络输出 [-1,1] → 旋翼推力指令（clip + 保存）
        self._actions = actions.clone().clamp(-1.0, 1.0)

    def _apply_action(self):
        # 4 旋翼推力 → 总推力(z) 与三轴力矩；细节见官方 quadcopter_env.py
        # self._thrust[:, 0, 2] = ... ; self._moment[:, 0, :] = ...
        self._robot.set_external_force_and_torque(self._thrust, self._moment)

    def _get_observations(self):
        # 目标相对位置 + 线速度 + 投影重力(姿态) + 角速度 → (num_envs, 12)
        ...
        # return {"policy": obs}

    def _get_rewards(self):
        # pos_err 平方惩罚 + 姿态(投影重力 xy 分量)惩罚 + 动作平方惩罚 + 存活奖励
        ...

    def _get_dones(self):
        # died = 触地或高度异常；time_out = 超过 episode_length_s
        ...

    def _reset_idx(self, env_ids):
        # 随机化初始位置/速度（域随机化入口），写回 self._robot.write_*_to_sim
        ...

## 5. 调试清单（自定义任务 90% 的坑在这里）

1. **先 1 个环境 + 渲染**跑通，再上 4096：`--num_envs=1`（去掉 `--headless`）肉眼检查
   spawn 位置、轴向、比例——USD 资产 Z-up/米制不匹配是最常见错误；
2. **观测形状**：`obs.shape == (num_envs, observation_space)`，dtype 全 float32，无 NaN
   （`torch.isnan(obs).any()` 打在第一轮 reset 后）；
3. **奖励量级**：各项 `weight × 典型值` 应在同一数量级；打印每个 RewTerm 的
   未加权均值（`env.reward_manager` 有 per-term 统计）；
4. **终止条件太松/太紧**：太松 → 策略学会「躺平吃存活奖励」；太紧 → episode 太短学不到东西。
   先看随机策略的 episode length 分布再定阈值；
5. **decimation 与 dt**：控制频率 = `1/(dt × decimation)`，无人机姿态环建议 ≥50 Hz；
6. **动作 scale**：网络输出 `tanh` 后是 [-1,1]，`scale` 决定物理量级——scale 过大导致振荡，
   过小导致「推不动」。

## 6. 训练与评估命令

```bash
# 训练（4096 环境）
./isaaclab.sh -p scripts/reinforcement_learning/rsl_rl/train.py \
    --task=My-Cartpole-v0 --num_envs=4096 --headless --max_iterations 500

# 少量环境回放验证（把 <run_dir> 换成 logs/rsl_rl/ 下实际目录）
./isaaclab.sh -p scripts/reinforcement_learning/rsl_rl/play.py \
    --task=My-Cartpole-v0 --num_envs=16 \
    --checkpoint logs/rsl_rl/<run_dir>/model_500.pt
# 预期输出：渲染窗口中摆杆保持竖直、小车居中；终端打印 episode 回报统计。
```

---

## ✏️ 练习

1. **奖励项消融设计**（★，约 20 分钟，纸面）
   为 Part A 的 CartPole 设计 3 组奖励消融实验（只改 `RewardsCfg` 的 weight），
   写出每组假设（哪个行为会变好/变坏）与判定指标。
   **交付物**：`journal/cartpole_reward_ablation.md`（表格：组别/权重/假设/指标）。
2. **精读官方 Quadcopter**（★★，约 60 分钟）
   阅读 Isaac Lab 源码 `direct/quadcopter/quadcopter_env.py`，补全本 notebook Part B 骨架中
   `_get_observations/_get_rewards/_get_dones` 三个方法，并注释每行张量运算的含义。
   **交付物**：`journal/quadcopter_annotated.py`。
3. **移植 DroneHoverEnv 的奖励配方**（★★，约 45 分钟）
   阶段一的奖励是 `-err² - 0.1·v² - 0.01·u²`。把它改写成 Isaac Lab 张量形式
   （输入 `env`，输出 `(num_envs,)` tensor），要求：err 为 3D 位置误差、v 为机体线速度、
   u 为 4 维旋翼指令；并说明各系数在 3D 情形下是否需要调整及理由。
   **交付物**：`journal/hover_reward.py`（含 `torch` 实现 + 100 字说明）。
4. **终止条件实验**（★★★，约 2 小时，需 GPU）
   在 Part A 任务上分别设置小车出界阈值 `±3.0 / ±1.5 / ±0.8`，各训练 200 iteration，
   记录收敛速度与最终 mean_reward，分析「终止条件松紧」对策略的影响。
   **交付物**：`journal/termination_experiment.md` + TensorBoard 截图描述。

## 参考答案

<details>
<summary>练习 1：奖励项消融设计（参考）</summary>

| 组别 | 改动 | 假设 | 判定指标 |
|------|------|------|----------|
| 基线 | alive 1.0, terminating -2.0, pole -1.0, cart -0.5, vel -1e-3 | — | 收敛 iteration、最终 mean_reward |
| A：去位置项 | cart_pos weight=0 | 收敛更快但小车漂移（只保杆不倒） | 训练中 `cart_pos` 均值是否增大 |
| B：重罚速度 | joint_vel weight=-0.05 | 动作更平滑但可能学保守（杆倒得慢但也扶得慢） | episode length、动作 L2 |
| C：高存活 | alive weight=5.0 | 风险：奖励 hacking——策略消极保活，不倒也不纠偏 | pole_angle 均值是否反而变差 |

判定建议固定随机种子、每组 ≥2 次重复，用 mean_reward 曲线 + 关键 term 均值双指标。
</details>

<details>
<summary>练习 2：精读官方 Quadcopter（要点提示）</summary>

- `_get_observations`：核心是 `self._robot.data.root_pos_w - self._desired_pos_w`（目标相对位置）、
  `root_lin_vel_b`（机体系线速度）、`projected_gravity_b`（用重力在机体系的投影表达姿态，
  避免四元数不连续）、`root_ang_vel_b`；
- `_get_rewards`：`pos_err` 用 `torch.linalg.norm(..., dim=1)`，姿态误差用投影重力的 xy 分量平方和；
  各项先 `exp(-k·err)` 或平方惩罚，再按 weight 求和；注意奖励每项都乘了 `step_dt`（官方做法）；
- `_get_dones`：`died = (z < 0) | (z 过高)` 或位置偏离阈值；`time_outs = episode_length_buf >= max_episode_length - 1`。

对照官方实现逐行注释即可，重点理解**全部运算是 `(num_envs, dim)` 批量张量，没有任何 for 循环**。
</details>

<details>
<summary>练习 3：张量化悬停奖励（参考实现）</summary>

```python
import torch
from isaaclab.envs import DirectRLEnv

def hover_reward(env: DirectRLEnv) -> torch.Tensor:
    # 3D 位置误差（目标在 self._desired_pos）
    pos_err = env._robot.data.root_pos_w - env._desired_pos  # (N, 3)
    err_sq = torch.sum(pos_err ** 2, dim=1)                  # (N,)
    vel_sq = torch.sum(env._robot.data.root_lin_vel_b ** 2, dim=1)
    act_sq = torch.sum(env._actions ** 2, dim=1)
    return -(err_sq + 0.1 * vel_sq + 0.01 * act_sq)
```

系数调整讨论：① 3D 下 err 是三个分量平方和，同等精度下数值约为一维的 3 倍，
可把位置项系数除以 3 或改用 `exp(-err²)` 有界化；② 3D 多了水平方向，
`0.1·v²` 可拆成垂直/水平不同权重（水平速度容忍度通常更高）；
③ `0.01·u²` 作用在 4 维动作上，量纲已归一化，可保持不变，但建议先观察训练中
各项未加权均值再定权重（见调试清单第 3 条）。
</details>

<details>
<summary>练习 4：终止条件实验（预期结论）</summary>

- `±3.0`（松）：episode 长、探索充分，但策略可能满足于「在轨道两端来回荡」，
  位置项收敛慢；
- `±1.5`（适中）：通常收敛最快且 final reward 最高（探索与约束平衡）；
- `±0.8`（紧）：早期大量 early termination，负样本多，训练前期 mean_reward 低且震荡；
  若最终收敛，策略往往更「居中」，但样本效率差。

分析框架：把「episode 平均长度曲线」和「mean_reward 曲线」放在一起看——
长度快速触底说明终止太紧；长度很长但 reward 不涨说明太松。
TensorBoard 中对应 `Episode/` 与 `Train/` 命名空间。
</details>

---

## 延伸阅读

- [Isaac Lab 文档：环境与任务教程](https://isaac-sim.github.io/IsaacLab/)（Tutorials → Environments / Tasks）
- [GitHub: isaac-sim/IsaacLab](https://github.com/isaac-sim/IsaacLab)（`source/isaaclab_tasks/` 下所有官方任务源码）
- [GitHub: isaac-sim/IsaacLabExtensionTemplate](https://github.com/isaac-sim/IsaacLabExtensionTemplate)（自定义任务扩展模板）
- [OpenUSD](https://openusd.org/)（资产侧问题回到 W31 的概念）